# S9 Foundation — Data, Features, Splits, Preprocessing

Shared substrate for scheme **S9** (LOWO over 20 waves + wave-19 holdout, wave 12 kept).

**Run all cells top-to-bottom**, then open `s9_content_based_model.ipynb` in the same kernel.

**Features:** `feature_engineering_experiment.ipynb` **E4_Full** set (134 columns): 89 raw `_A`/`_B` columns with **`gender_B` dropped**, plus 45 engineered columns (`diff_age`, pref-match, interest diffs, etc.). Built from keep-w12 parquet (**8,368 rows**, no row drops). **Not** from `two_sided_pair_features.csv` (global imputation). Each S9 split mean-imputes using **train waves only**.


## 1 — Paths and configuration

In [ ]:
import hashlib
import json
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics import ndcg_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
for candidate in [ROOT, *list(ROOT.parents)[:4]]:
    if (candidate / "preprocessing" / "speed_dating_clean_keep_w12.parquet").exists():
        ROOT = candidate
        break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

NOTEBOOKS_DIR = ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

KEEP_PARQUET = ROOT / "preprocessing" / "speed_dating_clean_keep_w12.parquet"
if not KEEP_PARQUET.exists():
    alt = ROOT / "Charlie" / "speed_dating_clean_keep_w12.parquet"
    if alt.exists():
        KEEP_PARQUET = alt

RESULTS = ROOT / "results"
(RESULTS / "runs").mkdir(parents=True, exist_ok=True)

SEED = 42
SCHEME = "S9"
HOLD_W19 = [19]

CONFIG = {
    "s1_model": "lr",
    "s1_C": 0.1,
    "seed": SEED,
    "scheme": SCHEME,
}

print(f"ROOT={ROOT}")
print(f"KEEP_PARQUET={KEEP_PARQUET} (exists={KEEP_PARQUET.exists()})")


## 2 — Provenance helpers

In [ ]:
def file_sha(path: Path, n_hex: int = 12) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:n_hex]


def df_fingerprint(df: pd.DataFrame) -> dict:
    return {
        "n_rows": int(len(df)),
        "n_waves": int(df["wave"].nunique()),
        "waves": sorted(int(w) for w in df["wave"].unique()),
        "dec_rate": round(float(df["dec"].mean()), 4),
        "match_rate": round(float(df["match"].mean()), 4),
    }


def params_hash(obj) -> str:
    return hashlib.sha256(
        json.dumps(obj, sort_keys=True, default=str).encode()
    ).hexdigest()[:12]


## 3 — Load keep-w12 and build E4_Full feature matrix

Same engineering as `feature_engineering_experiment.ipynb`: drop redundant `gender_B`, add groups 1–4 (`diff_age`, pref-match, …). Pair-level transforms only — no labels used.

In [ ]:
from s9_feature_engineering import build_s9_dataset, ID_COLS

if not KEEP_PARQUET.exists():
    raise FileNotFoundError(f"Missing cleaned parquet: {KEEP_PARQUET}")

_bundle = build_s9_dataset(KEEP_PARQUET)
DATA = _bundle["DATA"]
RAW_FEATURE_COLS = _bundle["RAW_FEATURE_COLS"]
EXCLUDED_FEATURE_COLS = _bundle["EXCLUDED_FEATURE_COLS"]
ENGINEERED_FEATURE_COLS = _bundle["ENGINEERED_FEATURE_COLS"]
S1_FEATS = _bundle["S1_FEATS"]
CB_FEATS = _bundle["CB_FEATS"]
THEME_MAP = _bundle["THEME_MAP"]
THEMES = _bundle["THEMES"]
FEATURE_GROUPS = _bundle["FEATURE_GROUPS"]

DATA_SHA = {"keep_parquet": file_sha(KEEP_PARQUET)}
fp = df_fingerprint(DATA)
raw_nan_rows = int(DATA[RAW_FEATURE_COLS].isna().any(axis=1).sum())
print(
    f"rows={fp['n_rows']:,}  waves={fp['n_waves']}  "
    f"dec={fp['dec_rate']}  match={fp['match_rate']}"
)
print(f"Excluded redundant: {EXCLUDED_FEATURE_COLS}")
print(f"Raw: {len(RAW_FEATURE_COLS)} | Engineered: {len(ENGINEERED_FEATURE_COLS)} | Total: {len(S1_FEATS)}")
print(f"Has diff_age: {'diff_age' in S1_FEATS} | gender_B in matrix: {'gender_B' in S1_FEATS}")
print(f"Rows with any raw NaN (filled per split): {raw_nan_rows}")


## 4 — Scheme S9 splits

In [ ]:
def scheme_splits(df: pd.DataFrame) -> list[dict]:
    waves = sorted(int(w) for w in df["wave"].unique())
    rot = [w for w in waves if w not in HOLD_W19]

    def lowo(rotation_waves, all_waves):
        return [
            {
                "tag": f"wave{w}",
                "kind": "lowo",
                "train_waves": [x for x in all_waves if x != w],
                "eval_waves": [w],
            }
            for w in rotation_waves
        ]

    out = lowo(rot, rot)
    out.append({
        "tag": "holdout_w19",
        "kind": "holdout",
        "train_waves": rot,
        "eval_waves": HOLD_W19,
    })
    return out


S9_SPLITS = scheme_splits(DATA)
waves = set(int(w) for w in DATA["wave"].unique())
for sp in S9_SPLITS:
    tr, ev = set(sp["train_waves"]), set(sp["eval_waves"])
    assert tr.isdisjoint(ev), sp["tag"]
    assert tr | ev <= waves, sp["tag"]
print(f"S9: {len(S9_SPLITS)} splits ({sum(1 for s in S9_SPLITS if s['kind']=='lowo')} LOWO + 1 holdout)")


## 5 — Leakage-safe preprocessor (train-fold mean impute)

Matches the prototype's `SimpleImputer(strategy='mean')` but statistics are fit on **train waves only**. Standard scaling for LR is applied inside `Stage1LR`.

In [ ]:
def fit_preprocessor(train_df: pd.DataFrame) -> dict:
    imputer = SimpleImputer(strategy="mean")
    imputer.fit(train_df[CB_FEATS])
    return {"imputer": imputer}


def apply_preprocessor(frame: pd.DataFrame, pp: dict) -> pd.DataFrame:
    out = frame.copy()
    out[CB_FEATS] = pp["imputer"].transform(out[CB_FEATS])
    return out


def process_split(df, train_waves, eval_waves):
    tr_raw = df[df["wave"].isin(train_waves)].copy()
    ev_raw = df[df["wave"].isin(eval_waves)].copy()
    pp = fit_preprocessor(tr_raw)
    return apply_preprocessor(tr_raw, pp), apply_preprocessor(ev_raw, pp), pp


_sp = S9_SPLITS[0]
_tr, _ev, _ = process_split(DATA, _sp["train_waves"], _sp["eval_waves"])
assert not _tr[S1_FEATS].isna().any().any(), "NaN in train features after impute"
assert not _ev[S1_FEATS].isna().any().any(), "NaN in eval features after impute"
print(f"Preprocessor OK: train={len(_tr):,} eval={len(_ev):,}")


## 6 — Evaluation metrics

In [ ]:
def mirror_col(df, col):
    lut = {(int(i), int(j)): v for i, j, v in zip(df["iid"], df["pid"], df[col])}
    return np.array([
        lut.get((int(j), int(i)), v)
        for i, j, v in zip(df["iid"], df["pid"], df[col])
    ])


def ndcg_at5(df, score_col):
    vals = []
    for _, g in df.groupby("iid"):
        if g["match"].sum() == 0:
            continue
        vals.append(ndcg_score(g[["match"]].T.values, g[[score_col]].T.values, k=5))
    return (float(np.mean(vals)) if vals else float("nan"), len(vals))


def mutual_match_at5(df, score_col, k=5):
    vals = []
    excluded = 0
    for _, g in df.groupby("iid"):
        matches = set(g.loc[g["match"] == 1, "pid"].astype(int))
        if not matches:
            excluded += 1
            continue
        top = g.nlargest(k, score_col)["pid"].astype(int).tolist()
        hits = sum(1 for pid in top if pid in matches)
        vals.append(hits / min(k, len(matches)))
    return (
        float(np.mean(vals)) if vals else float("nan"),
        len(vals),
        excluded,
    )


def add_reciprocal_scores(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    ab = out["score_cb"].to_numpy(float)
    ba = out["score_cb_rev"].to_numpy(float)
    denom = ab + ba
    hm = np.zeros_like(ab)
    mask = denom != 0
    hm[mask] = 2 * ab[mask] * ba[mask] / denom[mask]
    out["score_recip_hm"] = hm
    out["score_recip_gm"] = np.sqrt(ab * ba)
    return out


print("Metrics ready. Foundation complete.")


---

**Foundation complete.** Open `s9_content_based_model.ipynb` next (same kernel, or it will auto-load this notebook if needed).